### extract_emo_sentiment.ipynb

In this script, we will begin extracting emotional sentiment for songs we have (28 emotions total).

In [ ]:
import os 
import numpy as np 
import pandas as pd 
import csv
import matplotlib.pyplot as plt 
import seaborn as sns
import random 
sns.set(rc={'figure.figsize':(15, 8)})

from glob import glob
from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline # import NLP architecture
from scipy.sparse import dok_matrix
from sklearn.metrics.pairwise import cosine_similarity
import json

%autosave 300

In [ ]:
# Set up our directories 
currPath = os.getcwd()
playlistPath = os.path.abspath(os.path.join(currPath, os.pardir, 'data/playlist_subset'))
dataPath = os.path.join(playlistPath, 'lyrics')
lyrics_df = pd.read_csv(glob(os.path.join(dataPath, 'all_lyrics*'))[0])
lyrics_df = lyrics_df.drop(columns=['Unnamed: 0']) # get rid of unnecessary columns

# Also, set up our emotion classifier (from hugging face)
tokenizer = AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")
model = AutoModelForSequenceClassification.from_pretrained("SamLowe/roberta-base-go_emotions")
emotion_classifier = pipeline("text-classification", model=model, truncation=True, tokenizer=tokenizer, max_length=512, return_all_scores=True)

In [ ]:
# Create our json functions that will save out data later...  
def make_json_serializable(obj):
    if isinstance(obj, (np.integer, np.int_, np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float_, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (list, tuple)):
        return [make_json_serializable(i) for i in obj]
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif pd.isna(obj):  # handle NaNs or None
        return None
    else:
        return obj

def add_to_json(file_path, data_dict):
    serializable_dict = make_json_serializable(data_dict)
    
    # Ensure it ends with a newline and is one line per JSON object
    json_line = json.dumps(serializable_dict, ensure_ascii=False) + '\n'
    
    with open(file_path, 'a', encoding='utf-8') as f:
        f.write(json_line)

#### Get the sentiments for all the lyrics

In [ ]:
# For each song....
for song in lyrics_df.index.values: 
    # Get the lyrics for this song
    curr_lyrics = lyrics_df.loc[song, 'lyrics']
    lyric_moods = emotion_classifier(curr_lyrics)[0]

    # Then, save it to a dataframe and save the output
    mood_master = []
    for l in lyric_moods: 
        mood_name = list(l.values())[0]
        mood_score = list(l.values())[1]
        mood_df = pd.DataFrame([mood_score], index=[mood_name]).T
        mood_master.append(mood_df)

    # Then, concatenate the output and add to our dataframe 
    mood_master = pd.concat(mood_master, axis=1)

    # And finally, save the row to a new csv
    csv_row = pd.concat([pd.DataFrame(lyrics_df.loc[song]).T.reset_index(drop=True), mood_master], axis=1)
    with open(os.path.join(dataPath, 'lyrics_27_goemotions.csv'), "a", newline='') as this_file:
        writer = csv.writer(this_file, quoting=csv.QUOTE_MINIMAL)

        # Add headers if we are on the first row
        if song == 0:
            writer.writerow(csv_row.columns)

        # Add the data row
        writer.writerow(csv_row.values.flatten().tolist())

In [ ]:
# Load in the lyrics sentiment
emo_df = glob(os.path.join(dataPath, 'emo/*27*.csv')); assert len(emo_df) == 1
emo_df = pd.read_csv(emo_df[0])

# Remove the instrumental songs 
instr_percent = len(emo_df[emo_df['lyrics'] == 'Instrumental']) / len(emo_df); instr_percent = instr_percent * 100 
print("Portion of Instrumental songs we are excluding: %s%%" % round(instr_percent, 2))

# Get rid of the instrumental songs
emo_df = emo_df[emo_df['lyrics'] != 'Instrumental']

In [ ]:
# First, determine which emotions are the most common out of 27
# Define the columns we want to deal with
emo_cols = emo_df.columns[-28:]

meta_col = [] # create df for mean/variance/product
for x in emo_cols: 
    # Get the mean, variance and product of this column
    curr_col = emo_df[x]
    col_mean = np.average(curr_col); col_var = np.var(curr_col)

    col_df = pd.DataFrame([x, col_mean, col_var, col_mean*col_var], index=['emotion', 'mean', 'var', 'product']).T
    meta_col.append(col_df)

meta_col = pd.concat(meta_col)

In [ ]:
# Then, sort emotions by ascending to get the top ones 
meta_col = meta_col.sort_values(by='product',ascending = False)
meta_col.reset_index(drop=True, inplace=True)

# Plot the product to see what everything looks like
sns.barplot(meta_col, x="emotion", y="product")
# plt.show()

# Finally, choose the top 10 emotions (by removing the emotions we don't need)
emo_df = emo_df.drop(columns = meta_col[10:]['emotion'].values)


In [ ]:
# Plot the mean to see what everything looks like
sns.barplot(meta_col, x="emotion", y="mean")
# plt.show()

In [ ]:
# Plot the variance to see what everything looks like
sns.barplot(meta_col, x="emotion", y="var")
# plt.show()